In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
# List all CSV files in the 'datathon-2026-round-1' directory
csv_dir = 'datathon-2026-round-1'
csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]

# Load each CSV file into a dataframe with variable name as the file name (without .csv)
for file in csv_files:
    var_name = os.path.splitext(file)[0]
    df = pd.read_csv(os.path.join(csv_dir, file))
    globals()[var_name] = df

/tmp/ipykernel_220924/2926216130.py:8: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(csv_dir, file))


In [4]:
#Q1 
# orders = orders[orders['order_status'] == '']
multi_order = orders['customer_id'].value_counts()
multi_order = multi_order[multi_order > 1].index

orders_multi = orders[orders['customer_id'].isin(multi_order)].copy()

orders_multi['order_date'] = pd.to_datetime(orders_multi['order_date'])

orders_multi = orders_multi.sort_values(['customer_id', 'order_date'])

orders_multi['inter_order_gap'] = orders_multi.groupby('customer_id')['order_date'].diff().dt.days

median_gap = orders_multi['inter_order_gap'].median()
print(f"{median_gap:.0f}")

# Answer: 144

144


In [5]:
#Q2
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

segment_margin = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)

top_segment = segment_margin.idxmax()
top_margin = segment_margin.max()
print(f"Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: {top_segment} ({top_margin:.2%})")
print(segment_margin)
# Answer: Standard

Phân khúc có tỷ suất lợi nhuận gộp trung bình cao nhất: Standard (31.34%)
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64


In [6]:
#Q3
returns_streetwear = returns.merge(products[['product_id', 'category']], on='product_id')

returns_streetwear = returns_streetwear[returns_streetwear['category'] == 'Streetwear']
most_common_reason = returns_streetwear['return_reason'].value_counts().idxmax()
print("Lý do trả hàng nhiều nhất cho Streetwear:", most_common_reason)
print(returns_streetwear['return_reason'].value_counts())
# Answer: wrong_size

Lý do trả hàng nhiều nhất cho Streetwear: wrong_size
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64


In [7]:
#Q4
avg_bounce_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
lowest_bounce_source = avg_bounce_by_source.idxmin()
lowest_bounce_value = avg_bounce_by_source.min()
print(f"Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {lowest_bounce_source} ({lowest_bounce_value:.5f})")
avg_bounce_by_source
#Answer: email_campaign

Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: email_campaign (0.00446)


traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

In [8]:
#Q5
promo_applied_pct = order_items['promo_id'].notnull().mean() * 100
print(f"Tỷ lệ dòng áp dụng khuyến mãi: {promo_applied_pct:.2f}%")
#Answer: 39%

Tỷ lệ dòng áp dụng khuyến mãi: 38.66%


In [9]:
#Q6
valid_customers = customers[customers['age_group'].notnull()]

orders_per_customer = orders['customer_id'].value_counts()

valid_customers = valid_customers.copy()
valid_customers['order_count'] = valid_customers['customer_id'].map(orders_per_customer).fillna(0)

avg_orders_by_age = valid_customers.groupby('age_group')['order_count'].mean().sort_values(ascending=False)

top_age_group = avg_orders_by_age.idxmax()
top_avg_orders = avg_orders_by_age.max()
print(f"Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: {top_age_group} ({top_avg_orders:.2f})")
avg_orders_by_age

# Answer: 55+

Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: 55+ (5.41)


age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64

In [10]:
#Q7
sales_geo = order_items[["order_id", "unit_price", "quantity"]].copy()
sales_geo["sales"] = sales_geo["unit_price"] * sales_geo["quantity"]

# merge tables
sales_geo = sales_geo.merge(orders[["order_id", "zip"]], on="order_id", how="left")
sales_geo = sales_geo.merge(geography[["zip", "region"]], on="zip", how="left")
# sales_geo['city'] = sales_geo['city'].apply(no_accent_vietnamese).str.lower().str.strip().str.replace('tp. ', '').str.replace(' ', '').str.replace('thành phố ', '').str.replace('-', '').str.replace('city','')

# Calculate total sales by city
city_sales = sales_geo.groupby('region')['sales'].sum().reset_index()
city_sales.sort_values('sales', ascending=False)

# Answer: East

,region,sales
1,East,7.637533e+09
0,Central,4.941908e+09
2,West,3.851035e+09


In [11]:
#Q8
cancel_status = orders[orders['order_status'] == 'cancelled']
payment_method_counts = cancel_status['payment_method'].value_counts()
payment_method_counts

# Answer: credit card

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

In [12]:
# Q9
returns_with_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
order_items_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')

returns_by_size = returns_with_size.groupby('size').size()
order_items_by_size = order_items_with_size.groupby('size').size()

return_rate_by_size = (returns_by_size / order_items_by_size).sort_values(ascending=False)

print(return_rate_by_size)
print()
print(f"Highest return rate size: {return_rate_by_size.idxmax()}")

# Answer: size S

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64

Highest return rate size: S


In [13]:
# Q10
payments_installments = payments[payments['installments'] > 0]
avg_payment_by_plan = payments_installments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

top_plan = avg_payment_by_plan.idxmax()
top_value = avg_payment_by_plan.max()
print(f"Kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất: {top_plan} kỳ ({top_value:,.2f})")
avg_payment_by_plan
# Answer: 6

Kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất: 6 kỳ (24,446.65)


installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64